In [4]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df = pd.read_csv("phishing_urls_nlp_dataset.csv")

X_text = df['clean_url']
y = df['label']


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_chars = 200   # max URL length

tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts(X_text)

sequences = tokenizer.texts_to_sequences(X_text)
X_seq = pad_sequences(sequences, maxlen=max_chars, padding='post')


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Flatten, Dense, Dropout,Input

vocab_size = len(tokenizer.word_index) + 1

model = Sequential([
    Input(shape=(max_chars,)),
    Embedding(input_dim=vocab_size, output_dim=50),
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 50)        │        13,850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 196, 64)        │        16,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 98, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 94, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6016)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       770,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 841,694 (3.21 MB)

 Trainable params: 841,694 (3.21 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)


Epoch 1/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 158s 42ms/step - accuracy: 0.8762 - loss: 0.3303 - val_accuracy: 0.9186 - val_loss: 0.2081
Epoch 2/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 213s 45ms/step - accuracy: 0.9214 - loss: 0.2049 - val_accuracy: 0.9235 - val_loss: 0.1958
Epoch 3/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 170s 46ms/step - accuracy: 0.9309 - loss: 0.1768 - val_accuracy: 0.9324 - val_loss: 0.1781
Epoch 4/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 175s 48ms/step - accuracy: 0.9372 - loss: 0.1604 - val_accuracy: 0.9345 - val_loss: 0.1755
Epoch 5/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 168s 46ms/step - accuracy: 0.9408 - loss: 0.1496 - val_accuracy: 0.9375 - val_loss: 0.1630


In [7]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
predictions = model.predict(X_test)
y_pred = np.argmax(predictions, axis=1)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


4067/4067 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step
[[82925   204  2396    19]
 [  218 18967   101     5]
 [ 4871   119 13776    37]
 [   85     1   203  6210]]
              precision    recall  f1-score   support

           0       0.94      0.97      0.96     85544
           1       0.98      0.98      0.98     19291
           2       0.84      0.73      0.78     18803
           3       0.99      0.96      0.97      6499

    accuracy                           0.94    130137
   macro avg       0.94      0.91      0.92    130137
weighted avg       0.93      0.94      0.93    130137



In [8]:
import numpy as np
from sklearn.metrics import confusion_matrix

pred_cnn = model.predict(X_test)
y_pred_cnn = np.argmax(pred_cnn, axis=1)

cm_cnn = confusion_matrix(y_test, y_pred_cnn)
print(cm_cnn)


fn_cnn = {}

for i in range(cm_cnn.shape[0]):
    fn_cnn[f"Class_{i}"] = cm_cnn[i].sum() - cm_cnn[i][i]

print("CNN False Negatives per class:", fn_cnn)
print("Total CNN False Negatives:", sum(fn_cnn.values()))


4067/4067 ━━━━━━━━━━━━━━━━━━━━ 31s 8ms/step
[[82925   204  2396    19]
 [  218 18967   101     5]
 [ 4871   119 13776    37]
 [   85     1   203  6210]]
CNN False Negatives per class: {'Class_0': 2619, 'Class_1': 324, 'Class_2': 5027, 'Class_3': 289}
Total CNN False Negatives: 8259


In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input

vocab_size = len(tokenizer.word_index) + 1

lstm_model = Sequential([
    Input(shape=(max_chars,)),
    Embedding(vocab_size, 64),
    LSTM(128, return_sequences=False),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dense(4, activation='softmax')
])

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 200, 64)        │        17,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,060 (488.52 KB)

 Trainable params: 125,060 (488.52 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
class_weights


{0: 0.38032296528364307,
 1: 1.686467310309078,
 2: 1.7302821395806465,
 3: 5.00601054008309}

In [42]:
lstm_history = lstm_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    class_weight=class_weights
)


Epoch 1/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 1159s 317ms/step - accuracy: 0.5642 - loss: 0.8004 - val_accuracy: 0.6967 - val_loss: 0.6464
Epoch 2/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 1038s 283ms/step - accuracy: 0.7541 - loss: 0.4467 - val_accuracy: 0.8127 - val_loss: 0.4125
Epoch 3/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 968s 264ms/step - accuracy: 0.7927 - loss: 0.3768 - val_accuracy: 0.8188 - val_loss: 0.3933
Epoch 4/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 964s 263ms/step - accuracy: 0.8207 - loss: 0.3174 - val_accuracy: 0.8008 - val_loss: 0.4480
Epoch 5/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 967s 264ms/step - accuracy: 0.8438 - loss: 0.2747 - val_accuracy: 0.8662 - val_loss: 0.3107


In [43]:
from sklearn.metrics import classification_report, confusion_matrix

pred = lstm_model.predict(X_test)
y_pred_lstm = np.argmax(pred, axis=1)

print(confusion_matrix(y_test, y_pred_lstm))
print(classification_report(y_test, y_pred_lstm))


4067/4067 ━━━━━━━━━━━━━━━━━━━━ 163s 40ms/step
[[72200  1091 11721   532]
 [  254 18804   221    12]
 [ 2368   427 15773   235]
 [   60    22   238  6179]]
              precision    recall  f1-score   support

           0       0.96      0.84      0.90     85544
           1       0.92      0.97      0.95     19291
           2       0.56      0.84      0.67     18803
           3       0.89      0.95      0.92      6499

    accuracy                           0.87    130137
   macro avg       0.84      0.90      0.86    130137
weighted avg       0.90      0.87      0.88    130137



In [44]:
from tensorflow.keras.layers import Bidirectional

bilstm_model = Sequential([
    Input(shape=(max_chars,)),
    Embedding(vocab_size, 64),
    Bidirectional(LSTM(128)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dense(4, activation='softmax')
])

bilstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

bilstm_model.summary()


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 200, 64)        │        17,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 232,068 (906.52 KB)

 Trainable params: 232,068 (906.52 KB)

 Non-trainable params: 0 (0.00 B)

In [46]:
bilstm_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    class_weight=class_weights
)


Epoch 1/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 2275s 621ms/step - accuracy: 0.6945 - loss: 0.5899 - val_accuracy: 0.8069 - val_loss: 0.4473
Epoch 2/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 2328s 636ms/step - accuracy: 0.8092 - loss: 0.3428 - val_accuracy: 0.8257 - val_loss: 0.4020
Epoch 3/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 2383s 651ms/step - accuracy: 0.8403 - loss: 0.2788 - val_accuracy: 0.8356 - val_loss: 0.3550
Epoch 4/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 2134s 583ms/step - accuracy: 0.8584 - loss: 0.2439 - val_accuracy: 0.8644 - val_loss: 0.2996
Epoch 5/5
3661/3661 ━━━━━━━━━━━━━━━━━━━━ 1852s 506ms/step - accuracy: 0.8732 - loss: 0.2181 - val_accuracy: 0.8726 - val_loss: 0.2840


In [47]:

pred = bilstm_model.predict(X_test)
y_pred_bilstm = np.argmax(pred, axis=1)

print(confusion_matrix(y_test, y_pred_bilstm))
print(classification_report(y_test, y_pred_bilstm))


4067/4067 ━━━━━━━━━━━━━━━━━━━━ 216s 53ms/step
[[71790  1199 12022   533]
 [  109 18911   244    27]
 [ 1530   299 16722   252]
 [   19     1   195  6284]]
              precision    recall  f1-score   support

           0       0.98      0.84      0.90     85544
           1       0.93      0.98      0.95     19291
           2       0.57      0.89      0.70     18803
           3       0.89      0.97      0.92      6499

    accuracy                           0.87    130137
   macro avg       0.84      0.92      0.87    130137
weighted avg       0.91      0.87      0.88    130137

